# Provision the LEX project

This notebook does three things: (1) create LEX's own catalog schema and
project schema, both in LEX's dedicated database, (2) create LEX's
contract-specific tables (CONTRACT_REGISTER / CONTRACT_DOCUMENT_LINK /
CONTRACT_FIELD_EXTRACTS) and access role, and (3) deploy/redeploy the
Streamlit-in-Snowflake app. It intentionally does not sync, ingest, or
index documents, or link/extract contract data — that all happens later,
on demand, from the app itself, starting with uploading the Required
Contracts Register workbook on the Data Sources page.

Forked from the `project-llm-wiki` provisioning notebook template, which
normally centralizes its catalog in a shared database and ingests from
SharePoint via Microsoft Graph API. LEX follows neither part of the
template — everything it uses (catalog, data, Streamlit app) lives in
its own dedicated `MEDSCOMA` database and
`STREAMLIT_COMPUTE_POOL_CONTRACT_MGMT` compute pool with no shared
resources anywhere. LEX's contracts library is a genuine on-prem
network drive (SMB), but Snowflake can't reach it directly — documents
arrive via upload or the companion `lex_network_bridge` repo's
stage-pickup Task, not a direct SMB connection from this notebook or
app (see README's "Open items"). Run top to bottom the first time; most
cells are safe to re-run any time after that (each says so, or not, in
its own markdown).

In [ ]:
import sys
sys.path.insert(0, '../python')
from snowflake_session import get_session

session = get_session()
session.sql("USE ROLE ADVANCEDANALYTICS").collect()
session.sql("USE WAREHOUSE MTMWH02").collect()
session.sql("USE DATABASE MEDSCOMA").collect()
print('Connected.')

## One-time only: create LEX's own catalog schema
Skip this cell if `MEDSCOMA.APP_CATALOG` already exists (e.g. a re-run
after the first provisioning pass). Dedicated to LEX alone — nothing else
registers here. The `CREATE ... IF NOT EXISTS` statements make re-running
this harmless either way.

In [ ]:
from utils.sql_script import run_sql_file

count = run_sql_file(session, '../sql/00_setup_catalog.sql')
print(f'Catalog schema ready ({count} statements executed).')

## Create LEX's dedicated database + compute pool
LEX asks for full isolation — its own database (`MEDSCOMA`), holding its
catalog schema *and* its data schema, so nothing about LEX lives in or
depends on a database shared with any other project — plus its own
compute pool for container-runtime Streamlit (needed for a modern, pinned
Streamlit version and predictable resources against 100-500 page
OCR/indexing runs).

Both `CREATE DATABASE` and `CREATE COMPUTE POOL` are typically
`SYSADMIN`/`ACCOUNTADMIN`-only privileges that `ADVANCEDANALYTICS` doesn't
hold by default (the same gotcha this template's own README documents for
compute pools). This cell tries anyway and prints clear next steps if it
can't — if it fails, ask whoever holds `SYSADMIN` to run the two
`CREATE ...` statements printed in the error output, then re-run this cell
to confirm (it'll report "already exists" and move on). During build,
`QUERY_WAREHOUSE` stays `MTMWH02` (the shared build warehouse) — swap it to
a dedicated production warehouse later via `ALTER` on the project's catalog
row before productionisation.

In [ ]:
LEX_DATABASE = 'MEDSCOMA'
LEX_COMPUTE_POOL_NAME = 'STREAMLIT_COMPUTE_POOL_CONTRACT_MGMT'

def _try_create(sql_stmt, what):
    try:
        session.sql(sql_stmt).collect()
        print(f'OK  {what}')
    except Exception as e:
        print(f'SKIPPED — could not create {what} with the current role.')
        print(f'  Ask someone with SYSADMIN (or ACCOUNTADMIN) to run:')
        print(f'    {sql_stmt}')
        print(f'  Original error: {e}')

_try_create(
    f"""CREATE DATABASE IF NOT EXISTS {LEX_DATABASE}
        COMMENT = 'Dedicated database for LEX / Transition Contract Management RAG project'""",
    f'database {LEX_DATABASE}',
)
session.sql('USE WAREHOUSE MTMWH02').collect()
_try_create(
    f"""CREATE COMPUTE POOL IF NOT EXISTS {LEX_COMPUTE_POOL_NAME}
        MIN_NODES = 1 MAX_NODES = 2
        INSTANCE_FAMILY = CPU_X64_XS
        AUTO_SUSPEND_SECS = 300
        AUTO_RESUME = TRUE
        COMMENT = 'Dedicated compute pool for the LEX / Transition Contract Management Streamlit app'""",
    f'compute pool {LEX_COMPUTE_POOL_NAME}',
)

## Catalog migration: PROJECTS.NETWORK_DRIVE_* columns
Safe to re-run any time (idempotent `ADD COLUMN IF NOT EXISTS`). These columns are **not read by any code in this repo** — the in-app direct-SMB ingestion path was removed (see README's "Open items"). They exist purely as shared config storage for the companion `lex_network_bridge` repo, which queries this exact `PROJECTS` row directly (a plain SELECT, no Snowpark session) to get the real host/share/domain it connects to over SMB from inside the MTM network. **Do not drop these columns** — an earlier version of this cell did, which silently deleted that project's working bridge configuration with no error until the bridge tool's next run. Run this cell once on any catalog that's missing them; a brand-new catalog already has them from `00_setup_catalog.sql`'s `CREATE TABLE` and this is a no-op for it.

In [ ]:
catalog_migrations = [
    # PROJECTS.NETWORK_DRIVE_*: see the markdown above — these are
    # for the companion lex_network_bridge repo, not this app's own
    # code. NETWORK_DRIVE_SECRET_NAME is deliberately NOT restored —
    # it backed the removed in-app SMB feature's Snowflake SECRET
    # binding specifically, and nothing reads it any more (the bridge
    # tool gets its own SMB credentials from local environment
    # variables on the bridge host, never from this table).
    "ALTER TABLE MEDSCOMA.APP_CATALOG.PROJECTS ADD COLUMN IF NOT EXISTS NETWORK_DRIVE_HOST VARCHAR(255)",
    "ALTER TABLE MEDSCOMA.APP_CATALOG.PROJECTS ADD COLUMN IF NOT EXISTS NETWORK_DRIVE_SHARE VARCHAR(255)",
    "ALTER TABLE MEDSCOMA.APP_CATALOG.PROJECTS ADD COLUMN IF NOT EXISTS NETWORK_DRIVE_DEFAULT_PATH VARCHAR(1000)",
    "ALTER TABLE MEDSCOMA.APP_CATALOG.PROJECTS ADD COLUMN IF NOT EXISTS NETWORK_DRIVE_DOMAIN VARCHAR(100)",
]
for stmt in catalog_migrations:
    session.sql(stmt).collect()
    print(f"OK  {stmt}")

## Create the LEX project
Fill in the values below and run. Skip this cell if the project already exists (re-running errors on the duplicate project_code) — the `UPDATE` in the same cell still runs either way, so editing a setting below and re-running always takes effect even after the project exists.

**Confirm before running for real**: `CREATED_BY`. Left blank below on purpose rather than guessed.

In [ ]:
PROJECT_CODE = 'LEX'
PROJECT_NAME = 'LEX - Legal EXtraction & Contract Intelligence'
DESCRIPTION = 'Contract Q&A and automated stock-field extraction for Signed & Executed Contracts (MR5 Transition Contracts Team)'
CREATED_BY = ''                     # TODO: your name
QUERY_WAREHOUSE = 'MTMWH02'
COMPUTE_POOL = LEX_COMPUTE_POOL_NAME   # container runtime, unlike this template's warehouse-runtime default
DATA_DATABASE = LEX_DATABASE           # LEX's own dedicated database — no shared resources
SEGMENTATION_PROFILE = 'LEX_CONTRACT'  # clause/schedule-aware; see index_builder.py

# Retrieval mechanism config — see this template's own README's "How
# retrieval works" for what each one does/costs. Unlike the
# CALL CREATE_PROJECT(...) below (first-creation only), the UPDATE that
# applies these runs every time this cell runs — edit a value and re-run
# to change an already-existing project's settings.
SEGMENTATION_GRANULARITY = 'DETAILED'  # clause-level sections, not one per whole contract
ENABLE_RERANKING = True
ENABLE_VECTOR_SEARCH = True    # on from day one — summary-only routing is too coarse once
                                 # this project scales toward ~600 contracts
MAX_CANDIDATE_DOCS = 10
# Doubles as the per-chunk size for documents longer than this (see
# index_builder.py's chunked indexing) — sized well within typical Cortex
# model input limits per call, unlike a single-shot cap large enough to
# hold a whole 500-page contract, which would risk exceeding those limits
# outright rather than just chunking cleanly.
MAX_DOCUMENT_CHARS = 100000

existing = session.sql(
    "SELECT COUNT(*) AS C FROM MEDSCOMA.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()[0]["C"]

if existing > 0:
    print(f"Project '{PROJECT_CODE}' already exists — updating its settings below.")
else:
    result = session.sql(
        'CALL CREATE_PROJECT(?, ?, ?, ?, ?, ?, ?)',
        params=[PROJECT_CODE, PROJECT_NAME, DESCRIPTION, CREATED_BY,
                QUERY_WAREHOUSE, COMPUTE_POOL, DATA_DATABASE],
    ).collect()
    print(result[0][0])

session.sql(
    """UPDATE MEDSCOMA.APP_CATALOG.PROJECTS
       SET SEGMENTATION_PROFILE = ?, SEGMENTATION_GRANULARITY = ?,
           ENABLE_RERANKING = ?, ENABLE_VECTOR_SEARCH = ?, MAX_CANDIDATE_DOCS = ?,
           MAX_DOCUMENT_CHARS = ?
       WHERE PROJECT_CODE = ?""",
    params=[SEGMENTATION_PROFILE, SEGMENTATION_GRANULARITY,
            ENABLE_RERANKING, ENABLE_VECTOR_SEARCH, MAX_CANDIDATE_DOCS,
            MAX_DOCUMENT_CHARS, PROJECT_CODE],
).collect()
print(f"Settings applied: profile={SEGMENTATION_PROFILE}, granularity={SEGMENTATION_GRANULARITY}, "
      f"reranking={ENABLE_RERANKING}, vector_search={ENABLE_VECTOR_SEARCH}, "
      f"max_docs={MAX_CANDIDATE_DOCS}, max_document_chars={MAX_DOCUMENT_CHARS}")

## Set LEX's network drive location (for the companion bridge tool)
Not used by any code in this repo — this is shared config storage the companion `lex_network_bridge` repo reads directly from this `PROJECTS` row to know which SMB host/share to connect to. Safe to re-run any time; only updates when a value below is actually set. Leave a field as `None` to leave that column unchanged.

In [ ]:
LEX_NETWORK_DRIVE_HOST = None          # e.g. 'MTADFS201V.metrotrains.local'
LEX_NETWORK_DRIVE_SHARE = None         # e.g. 'apps$'
LEX_NETWORK_DRIVE_DEFAULT_PATH = None  # optional, e.g. 'Signed Contracts'
LEX_NETWORK_DRIVE_DOMAIN = None        # optional NTLM domain, e.g. 'METROTRAINS'

_fields = {
    'NETWORK_DRIVE_HOST': LEX_NETWORK_DRIVE_HOST,
    'NETWORK_DRIVE_SHARE': LEX_NETWORK_DRIVE_SHARE,
    'NETWORK_DRIVE_DEFAULT_PATH': LEX_NETWORK_DRIVE_DEFAULT_PATH,
    'NETWORK_DRIVE_DOMAIN': LEX_NETWORK_DRIVE_DOMAIN,
}
_to_set = {k: v for k, v in _fields.items() if v is not None}

if _to_set:
    set_clause = ', '.join(f'{col} = ?' for col in _to_set)
    session.sql(
        f"UPDATE MEDSCOMA.APP_CATALOG.PROJECTS SET {set_clause} WHERE PROJECT_CODE = ?",
        params=[*_to_set.values(), PROJECT_CODE],
    ).collect()
    print(f"Updated: {', '.join(_to_set.keys())}")
else:
    print('Nothing set — all fields are None. Fill in at least '
          'NETWORK_DRIVE_HOST/NETWORK_DRIVE_SHARE and re-run.')


## Create LEX's contract tables
`CONTRACT_REGISTER`, `CONTRACT_DOCUMENT_LINK`, and `CONTRACT_FIELD_EXTRACTS`
are LEX-specific — not part of the generic project-llm-wiki shape
`CREATE_PROJECT` already created (`RAW_DOCUMENTS`/`DOCUMENT_INDEX`). See
`sql/03_lex_contract_tables.sql` for the reference copy of this DDL. Safe
to re-run any time (idempotent `CREATE TABLE IF NOT EXISTS`).

In [ ]:
proj_row = session.sql(
    "SELECT DATA_DATABASE, DATA_SCHEMA FROM MEDSCOMA.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()
if not proj_row:
    raise ValueError(f"No project found with code '{PROJECT_CODE}' — run the project-creation cell first.")
qualified_schema = f"{proj_row[0]['DATA_DATABASE']}.{proj_row[0]['DATA_SCHEMA']}"

contract_table_ddl = [
    f"""CREATE TABLE IF NOT EXISTS {qualified_schema}.CONTRACT_REGISTER (
          CONTRACT_ID       INT IDENTITY PRIMARY KEY,
          CW_NUMBER          VARCHAR(50) NOT NULL UNIQUE,
          CONTRACT_TITLE      VARCHAR(500),
          STATUS               VARCHAR(20) DEFAULT 'ACTIVE',
          OVERVIEW_SUMMARY      VARCHAR(4000),
          OVERVIEW_GENERATED_AT  TIMESTAMP_NTZ,
          RECOMMENDED_ACTIONS        VARIANT,
          CLASSIFICATION_SCORECARD    VARIANT,
          CREATED_AT                    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
        )""",
    f"""CREATE TABLE IF NOT EXISTS {qualified_schema}.CONTRACT_DOCUMENT_LINK (
          LINK_ID          INT IDENTITY PRIMARY KEY,
          CONTRACT_ID       INT NOT NULL REFERENCES {qualified_schema}.CONTRACT_REGISTER(CONTRACT_ID),
          DOC_ID             INT NOT NULL REFERENCES {qualified_schema}.RAW_DOCUMENTS(DOC_ID),
          DOC_ROLE            VARCHAR(20) NOT NULL,
          EFFECTIVE_DATE       DATE,
          SEQUENCE_NO           INT,
          LINKED_BY              VARCHAR(200),
          LINKED_AT               TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
          UNIQUE (CONTRACT_ID, DOC_ID)
        )""",
    f"""CREATE TABLE IF NOT EXISTS {qualified_schema}.CONTRACT_FIELD_EXTRACTS (
          EXTRACT_ID        INT IDENTITY PRIMARY KEY,
          CONTRACT_ID        INT NOT NULL REFERENCES {qualified_schema}.CONTRACT_REGISTER(CONTRACT_ID),
          FIELD_KEY           VARCHAR(50) NOT NULL,
          FIELD_VALUE           VARCHAR(4000),
          SOURCE_DOC_ID           INT REFERENCES {qualified_schema}.RAW_DOCUMENTS(DOC_ID),
          SOURCE_NODE_ID           INT REFERENCES {qualified_schema}.DOCUMENT_INDEX(NODE_ID),
          SOURCE_QUOTE               VARCHAR(4000),
          HIGHLIGHT_PHRASE            VARCHAR(500),
          CONFIDENCE                   VARCHAR(20),
          MODEL_USED                    VARCHAR(50),
          EXTRACTED_AT                   TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
          IS_VERIFIED                     BOOLEAN DEFAULT FALSE,
          VERIFIED_BY                      VARCHAR(200),
          VERIFIED_AT                       TIMESTAMP_NTZ,
          UNIQUE (CONTRACT_ID, FIELD_KEY)
        )""",
    f"""CREATE STAGE IF NOT EXISTS {qualified_schema}.CONTRACT_OUTPUT_STAGE
          ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')""",
]
for stmt in contract_table_ddl:
    session.sql(stmt).collect()
print(f'Contract tables + output stage ready in {qualified_schema}.')

## Deploy the Streamlit app
Safe to re-run any time — re-stages every file with `overwrite=True` and
redeploys the app object with `CREATE OR REPLACE`. `environment.yml` is
staged at the stage **root** (not nested), since Streamlit-in-Snowflake
only reads it from there.

In [ ]:
import os

proj = session.sql(
    "SELECT * FROM MEDSCOMA.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()
if not proj:
    raise ValueError(f"No project found with code '{PROJECT_CODE}' — run the project-creation cell first.")
p = proj[0]

STREAMLIT_STAGE = f"MEDSCOMA.APP_CATALOG.{p['STREAMLIT_STAGE_NAME']}"
STREAMLIT_APP_NAME = f"MEDSCOMA.APP_CATALOG.{p['STREAMLIT_APP_NAME']}"
APP_QUERY_WAREHOUSE = p['QUERY_WAREHOUSE']

APP_COMPUTE_POOL = p['COMPUTE_POOL']
if not APP_COMPUTE_POOL or str(APP_COMPUTE_POOL).strip().lower() in ("", "none", "null"):
    APP_COMPUTE_POOL = None

session.sql(f"CREATE STAGE IF NOT EXISTS {STREAMLIT_STAGE}").collect()

# python/ keeps its own folder structure (python/config.py -> @stage/python/config.py),
# a sibling of Chat.py at the stage root. streamlit/'s contents are staged
# FLATTENED to the stage root (streamlit/Chat.py -> @stage/Chat.py,
# streamlit/pages/1_Data_Sources.py -> @stage/pages/1_Data_Sources.py) — a
# nested MAIN_FILE reliably fails to load on this account.
for root, _, files in os.walk("../python"):
    rel_root = os.path.relpath(root, "..")
    stage_dir = f"@{STREAMLIT_STAGE}/{rel_root}"
    for fname in files:
        if fname.endswith(".py"):
            session.file.put(f"{root}/{fname}", stage_dir,
                              auto_compress=False, overwrite=True)

for root, _, files in os.walk("../streamlit"):
    rel_root = os.path.relpath(root, "../streamlit")   # "." or "pages"
    stage_dir = f"@{STREAMLIT_STAGE}" if rel_root == "." else f"@{STREAMLIT_STAGE}/{rel_root}"
    for fname in files:
        if fname.endswith(".py"):
            session.file.put(f"{root}/{fname}", stage_dir,
                              auto_compress=False, overwrite=True)

# assets/ (the Contract Workspace Summary Template .docx) is staged as its
# own top-level folder too, a sibling of python/ — docx_report.py resolves
# it via os.path.dirname(__file__)/../assets/..., which on the stage means
# @stage/python/../assets/... = @stage/assets/..., so this folder name and
# nesting level both matter, not just "getting the file onto the stage
# somewhere".
for root, _, files in os.walk("../assets"):
    rel_root = os.path.relpath(root, "..")
    stage_dir = f"@{STREAMLIT_STAGE}/{rel_root}"
    for fname in files:
        session.file.put(f"{root}/{fname}", stage_dir,
                          auto_compress=False, overwrite=True)

# Stage EXACTLY ONE dependency manifest, matching the runtime this project is
# actually configured for. Staging both at once is ambiguous.
if APP_COMPUTE_POOL:
    manifest_to_stage = "pyproject.toml"
    manifest_to_remove = "environment.yml"
else:
    manifest_to_stage = "environment.yml"
    manifest_to_remove = "pyproject.toml"

local_path = f"../streamlit/{manifest_to_stage}"
if os.path.exists(local_path):
    session.file.put(local_path, f"@{STREAMLIT_STAGE}",
                      auto_compress=False, overwrite=True)

session.sql(f"REMOVE @{STREAMLIT_STAGE}/{manifest_to_remove}").collect()

# FROM (not the legacy ROOT_LOCATION) — required on accounts where
# ROOT_LOCATION has been retired for new/replaced Streamlit apps.
create_stmt = f"""
    CREATE OR REPLACE STREAMLIT {STREAMLIT_APP_NAME}
      FROM '@{STREAMLIT_STAGE}'
      MAIN_FILE = 'Chat.py'
      QUERY_WAREHOUSE = {APP_QUERY_WAREHOUSE}
"""
# PYPI_ACCESS_INTEGRATION is account-level plumbing that allow-lists
# pypi.org for `pip install` on container runtime (needed to resolve
# python-docx/reportlab) — holds no secrets or project data, and is
# pre-existing infra this repo doesn't create, so referencing it isn't a
# "shared resource" dependency the way a shared catalog/secret would be.
# Not needed on warehouse runtime, which resolves packages from
# Snowflake's own Anaconda channel instead of PyPI.
if APP_COMPUTE_POOL:
    create_stmt += f"""
      RUNTIME_NAME = 'SYSTEM$ST_CONTAINER_RUNTIME_PY3_11'
      COMPUTE_POOL = '{APP_COMPUTE_POOL}'
      EXTERNAL_ACCESS_INTEGRATIONS = (PYPI_ACCESS_INTEGRATION)
    """
else:
    create_stmt += """
      RUNTIME_NAME = 'SYSTEM$WAREHOUSE_RUNTIME'
    """

session.sql(create_stmt).collect()

print(f"Streamlit app deployed: {STREAMLIT_APP_NAME}")
print(f"  Warehouse: {APP_QUERY_WAREHOUSE}")
print(f"  Runtime:   {'container (' + APP_COMPUTE_POOL + ')' if APP_COMPUTE_POOL else 'warehouse'}")
print(f"  Manifest:  {manifest_to_stage} (removed {manifest_to_remove} if present)")
print("  Run the 'Restrict access' cell next to grant LEX_USERS USAGE on this app.")
print("  Then run the 'Set up the stage pickup task' cell to start "
      "automatically ingesting files staged by the bridge tool.")

## Set up the stage pickup task
Runs `sql/04_stage_pickup_task.sql` — the scheduled Task that drains `NETWORK_DRIVE_INBOX_STAGE` (filled by the companion `lex_network_bridge` repo) into `RAW_DOCUMENTS`, links each file to its contract, indexes it, and runs extraction, on a 5-minute schedule. Safe to re-run any time — every statement in that file is idempotent (`CREATE ... IF NOT EXISTS`/`CREATE OR REPLACE`). Must run **after** the deploy cell above.

This cell also zips up `python/` and stages it as `lex_python_imports.zip`, which the stored procedure's `IMPORTS` clause references — a bare stage *directory* reference (`@LEX_APP_STAGE/python/`) was tried first and **confirmed broken** on a live account (`ModuleNotFoundError: No module named 'ingestion'` — a directory IMPORTS doesn't flatten onto `sys.path` the way Streamlit's own `sys.path.insert()` does). A zip is Snowflake's documented, reliable way to import a multi-file/multi-package Python tree into a stored procedure.

In [ ]:
import io
import os
import zipfile
from utils.sql_script import run_sql_file

# Zip python/'s contents at the zip ROOT (config.py, ingestion/stage_pickup.py,
# utils/cortex_client.py, ...) — no extra 'python/' folder inside the zip —
# so imports inside the stored procedure (import config, from ingestion
# import stage_pickup) match exactly what Streamlit's own sys.path.insert()
# already makes work for the app itself.
zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk('../python'):
        for fname in files:
            if fname.endswith('.py'):
                full_path = os.path.join(root, fname)
                arcname = os.path.relpath(full_path, '../python')
                zf.write(full_path, arcname)
zip_buffer.seek(0)

IMPORTS_ZIP_STAGE_PATH = 'MEDSCOMA.APP_CATALOG.LEX_APP_STAGE/lex_python_imports.zip'
session.file.put_stream(zip_buffer, f'@{IMPORTS_ZIP_STAGE_PATH}',
                        auto_compress=False, overwrite=True)
print(f'Staged {IMPORTS_ZIP_STAGE_PATH} for the stored procedure\'s IMPORTS clause.')

n = run_sql_file(session, '../sql/04_stage_pickup_task.sql')
print(f"Stage pickup task set up ({n} statement(s) run).")
print("Manual run (don't wait for the schedule): "
      "CALL MEDSCOMA.APP_CATALOG.RUN_LEX_STAGE_PICKUP();")
print("History: SELECT * FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY("
      "TASK_NAME => 'LEX_STAGE_PICKUP_TASK')) ORDER BY SCHEDULED_TIME DESC;")

## Restrict access: the `LEX_USERS` role
Runs *after* deploying the app (above), not before — `GRANT USAGE ON
STREAMLIT ...` needs the Streamlit object to already exist, and it's only
created by the deploy cell. Safe to re-run any time.

Access control lives entirely at the Snowflake grant, not in application
code — `Chat.py` has no login screen. `LEX_USERS` gets `USAGE` on the
Streamlit app, compute pool, warehouse, and data schema, then is granted
once to `ADVANCEDANALYTICS` via Snowflake's role hierarchy — anyone
provisioned into `ADVANCEDANALYTICS` (managed externally via a security
group, not per-user `GRANT ROLE ... TO USER` statements here) inherits
`LEX_USERS`' privileges automatically. **This grants LEX access to
everyone who holds `ADVANCEDANALYTICS`, not just a named handful** — if
that role is broadly held on this account (its own doc comments describe
it as "the same service role every project in this catalog runs as"),
confirm that's the intended population before running this cell for
real.

`CREATE ROLE` is typically `SYSADMIN`/`ACCOUNTADMIN`-only — if
`ADVANCEDANALYTICS` doesn't hold it, this cell prints the exact statement
to hand to an admin, then skips the grants that depend on the role
existing (re-run once it's been created).

In [ ]:
app_row = session.sql(
    "SELECT STREAMLIT_APP_NAME, COMPUTE_POOL, QUERY_WAREHOUSE, DATA_DATABASE, DATA_SCHEMA "
    "FROM MEDSCOMA.APP_CATALOG.PROJECTS WHERE PROJECT_CODE = ?",
    params=[PROJECT_CODE],
).collect()[0]

role_exists = bool(session.sql("SHOW ROLES LIKE 'LEX_USERS'").collect())
if not role_exists:
    try:
        session.sql("CREATE ROLE IF NOT EXISTS LEX_USERS").collect()
        role_exists = True
        print('OK  role LEX_USERS')
    except Exception as e:
        print('SKIPPED — could not create role LEX_USERS with the current role.')
        print('  Ask someone with SYSADMIN (or ACCOUNTADMIN) to run:')
        print('    CREATE ROLE IF NOT EXISTS LEX_USERS')
        print(f'  Original error: {e}')

if role_exists:
    session.sql(
        f"GRANT USAGE ON STREAMLIT MEDSCOMA.APP_CATALOG.{app_row['STREAMLIT_APP_NAME']} TO ROLE LEX_USERS"
    ).collect()
    if app_row["COMPUTE_POOL"]:
        session.sql(f"GRANT USAGE ON COMPUTE POOL {app_row['COMPUTE_POOL']} TO ROLE LEX_USERS").collect()
    session.sql(f"GRANT USAGE ON WAREHOUSE {app_row['QUERY_WAREHOUSE']} TO ROLE LEX_USERS").collect()
    session.sql(f"GRANT USAGE ON DATABASE {app_row['DATA_DATABASE']} TO ROLE LEX_USERS").collect()
    session.sql(f"GRANT USAGE ON SCHEMA {app_row['DATA_DATABASE']}.{app_row['DATA_SCHEMA']} TO ROLE LEX_USERS").collect()

    # Access is managed via a security group, not individual named users:
    # LEX_USERS is granted once to ADVANCEDANALYTICS, and whoever that role's
    # own security-group membership provisions inherits LEX_USERS' privileges
    # automatically through Snowflake's role hierarchy. Re-running this grant
    # is a harmless no-op if it's already in place.
    session.sql("GRANT ROLE LEX_USERS TO ROLE ADVANCEDANALYTICS").collect()
    print('LEX_USERS role + object grants are in place, and granted to ROLE ADVANCEDANALYTICS — '
          'access follows that role\'s own security-group membership.')
else:
    print('Re-run this cell once LEX_USERS has been created by someone with SYSADMIN/ACCOUNTADMIN.')

## Schema migrations (safe to re-run any time)

New columns added to `RAW_DOCUMENTS`/`DOCUMENT_INDEX`/`PROJECTS`/LEX's own
contract tables after a project was first created won't retroactively
appear in its already-existing schema — the project-creation and
contract-tables cells above only run `CREATE ... IF NOT EXISTS`, which
does nothing to a table that already exists in an older shape. This cell
is the running list of forward-only, idempotent `ALTER TABLE ... ADD
COLUMN IF NOT EXISTS` statements for LEX's schema specifically, so picking
up a schema change is "re-run this cell" rather than a manual one-off
`ALTER TABLE` typed into a worksheet — the same discipline this template's
own equivalent cell uses elsewhere. A project provisioned for the first time with
the current version of this notebook already has every column from the
`CREATE TABLE` statements above and these are no-ops for it; they only
matter for a LEX project that existed before this cell's entries were
added.

In [ ]:
qualified_schema = f"{proj_row[0]['DATA_DATABASE']}.{proj_row[0]['DATA_SCHEMA']}"

# RAW_DOCUMENTS.SHAREPOINT_ITEM_ID -> SOURCE_ITEM_ID: a rename, not an ADD
# COLUMN, so it needs its own try/except rather than joining the
# IF-NOT-EXISTS list below (ALTER TABLE ... RENAME COLUMN has no IF EXISTS
# guard). No-op (prints and moves on) for a project whose RAW_DOCUMENTS
# table was created after this rename, which never had the old column.
try:
    session.sql(
        f"ALTER TABLE {qualified_schema}.RAW_DOCUMENTS RENAME COLUMN SHAREPOINT_ITEM_ID TO SOURCE_ITEM_ID"
    ).collect()
    print(f"OK  renamed {qualified_schema}.RAW_DOCUMENTS.SHAREPOINT_ITEM_ID -> SOURCE_ITEM_ID")
except Exception as e:
    print(f"SKIPPED renaming SHAREPOINT_ITEM_ID -> SOURCE_ITEM_ID (already applied, or column "
          f"never existed on this project): {e}")

migrations = [
    # SOURCE_ITEM_ID widened 200 -> 1000: UNC paths run longer than the
    # old SharePoint item-id column was sized for. COLLATE 'en-ci' is
    # required alongside SET DATA TYPE on this account even for a pure
    # length widen — see the SOURCE_QUOTE migration below for why.
    f"""ALTER TABLE {qualified_schema}.RAW_DOCUMENTS
        ALTER COLUMN SOURCE_ITEM_ID SET DATA TYPE VARCHAR(1000) COLLATE 'en-ci'""",
    # (contract-lookup UX) OVERVIEW_SUMMARY: the template's "Executive
    # Assessment" narrative, shown at the top of Contract Lookup and in
    # the .docx export — see contract_extraction.generate_contract_overview.
    f"ALTER TABLE {qualified_schema}.CONTRACT_REGISTER ADD COLUMN IF NOT EXISTS OVERVIEW_SUMMARY VARCHAR(4000)",
    f"ALTER TABLE {qualified_schema}.CONTRACT_REGISTER ADD COLUMN IF NOT EXISTS OVERVIEW_GENERATED_AT TIMESTAMP_NTZ",
    # (Contract Workspace Summary Template adoption) RECOMMENDED_ACTIONS /
    # CLASSIFICATION_SCORECARD: the template's "Recommended Actions" bullet
    # list (JSON array) and "Consolidated Procurement Assessment" scorecard
    # (JSON object) — see contract_extraction.generate_recommended_actions /
    # generate_classification_scorecard.
    f"ALTER TABLE {qualified_schema}.CONTRACT_REGISTER ADD COLUMN IF NOT EXISTS RECOMMENDED_ACTIONS VARIANT",
    f"ALTER TABLE {qualified_schema}.CONTRACT_REGISTER ADD COLUMN IF NOT EXISTS CLASSIFICATION_SCORECARD VARIANT",
    # (citation viewer) HIGHLIGHT_PHRASE: the short exact phrase the
    # citation panel highlights/searches for, verified as a substring of
    # SOURCE_QUOTE at extraction time — see contract_extraction.py's
    # _extract_highlight_phrase.
    f"ALTER TABLE {qualified_schema}.CONTRACT_FIELD_EXTRACTS ADD COLUMN IF NOT EXISTS HIGHLIGHT_PHRASE VARCHAR(500)",
    # SOURCE_QUOTE widened 2000 -> 4000: it now holds the cited section's
    # full excerpt (for the citation panel's exact-text view), not just a
    # short snippet. COLLATE 'en-ci' is required alongside SET DATA TYPE on
    # this account even for a pure length widen with no other type change —
    # ALTER COLUMN must match the existing column's collation exactly, and
    # this account applies 'en-ci' as the default VARCHAR collation (the
    # same gotcha hit widening NODE_SUMMARY in another project-llm-wiki
    # project's notebook).
    f"""ALTER TABLE {qualified_schema}.CONTRACT_FIELD_EXTRACTS
        ALTER COLUMN SOURCE_QUOTE SET DATA TYPE VARCHAR(4000) COLLATE 'en-ci'""",
]


# CONTRACT_OUTPUT_STAGE: cached Word/PDF Contract Workspace Summary
# outputs (see python/contract_output_cache.py) — added after this
# project may already have been provisioned, so a project created before
# this feature existed needs it added here rather than only in the
# "Create LEX's contract tables" cell above.
session.sql(
    f"""CREATE STAGE IF NOT EXISTS {qualified_schema}.CONTRACT_OUTPUT_STAGE
        ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')"""
).collect()
print(f"OK  CONTRACT_OUTPUT_STAGE ready in {qualified_schema}")

for stmt in migrations:
    session.sql(stmt).collect()
    print(f"OK  {stmt}")

## If the app still won't load
Snowsight's "Something went wrong" card only shows the top-level exception,
not a traceback. Run the cell below in *this* notebook (same Python
environment the app runs in) to import every current app dependency one at
a time and print a full traceback for whichever one actually fails.

In [ ]:
import sys, traceback

for p in ("../python", "../streamlit"):
    if p not in sys.path:
        sys.path.insert(0, p)

# Same import order Chat.py exercises, split out module by module so
# whichever one fails prints its own full traceback instead of one opaque
# top-level error.
modules_to_test = [
    "pandas",
    "config",
    "snowflake_session",
    "query_engine",
    "contract_linking",
    "contract_extraction",
    "required_contracts",
    "citation_viewer",
    "citation_panel_ui",
    "docx_report",
    "pdf_report",
    "contract_output_cache",
    "docx",
    "reportlab",
    "utils.cortex_client",
    "utils.sql_utils",
    "utils.sql_script",
    "utils.logging_utils",
    "ingestion.xlsx_parser",
    "ingestion.file_ingest",
    "ingestion.stage_pickup",
    "ingestion.index_builder",
]

for m in modules_to_test:
    try:
        __import__(m)
        print(f"OK    {m}")
    except Exception:
        print(f"FAILED {m}")
        traceback.print_exc()
        print()

print("\nPython:", sys.version)

## Debug: test JSON parsing on one document

If **Index new/unindexed documents** in the app fails with a Cortex
JSON-parsing error (`complete_json` in `python/utils/cortex_client.py`),
use this cell to reproduce it directly against one live document — seconds,
not a full Streamlit redeploy-and-click cycle. It calls the real
`complete`/`complete_json` functions from this repo, so a fix verified
here is testing the actual code path the app uses.

Makes two live Cortex calls — fine for interactive debugging, but don't
loop this over many documents; use the app's own indexing for that.

In [ ]:
import sys
if "../python" not in sys.path:
    sys.path.insert(0, "../python")

from config import load_project
from utils.cortex_client import complete, complete_json
from ingestion.index_builder import PROMPTS

DEBUG_PROJECT_CODE = PROJECT_CODE  # from the project-creation cell above
DEBUG_DOC_ID = None  # a specific RAW_DOCUMENTS.DOC_ID, or None for "first document"

project = load_project(session, DEBUG_PROJECT_CODE)
schema = project.qualified_schema
prompt_template = PROMPTS.get(project.segmentation_profile, PROMPTS["GENERIC"])

where = "DOC_ID = ?" if DEBUG_DOC_ID else "1=1"
params = [DEBUG_DOC_ID] if DEBUG_DOC_ID else []
doc = session.sql(
    f"SELECT DOC_ID, FILE_NAME, RAW_TEXT FROM {schema}.RAW_DOCUMENTS "
    f"WHERE {where} ORDER BY DOC_ID LIMIT 1",
    params=params,
).collect()[0]

print(f"Testing DOC_ID={doc['DOC_ID']} FILE_NAME={doc['FILE_NAME']!r}")
# Uses only the first chunk's worth of text — matches what
# index_builder._index_one_document sends in its first (and, for most
# documents, only) indexing call; see that function for the full chunked
# path used on documents longer than project.max_document_chars.
text = doc["RAW_TEXT"][: project.max_document_chars]
prompt = prompt_template.format(text=text, granularity_instruction="")

print("\n--- Step 1: raw Cortex response (full text) ---")
raw = complete(session, project.active_model, prompt)
print(raw)

print("\n--- Step 2: complete_json() result ---")
try:
    result = complete_json(session, project.active_model, prompt)
    print("Parsed OK. Keys:", list(result.keys()))
    print("document_summary:", result.get("document_summary", "")[:200])
    print("sections found:", len(result.get("sections", [])))
except Exception as e:
    print(f"FAILED: {type(e).__name__}: {e}")

## Next step
Open the Streamlit app and, on **Data Sources**: (1) upload the Required
Contracts Register workbook (the list of CW numbers currently in scope —
2 today, growing toward 8 for Build/validation) under its own tab, then
(2) ingest each contract's signed/executed PDF (rarely DOCX). Once a
contract and any of its variations/extensions are ingested, go to
**Contract Register** to link them into one family and run extraction —
after that, look the contract up on **Contract Lookup** (the app's landing
page) to see its standard questions answered laid out the same way as the
Contract Workspace Summary Template, click through to the cited passage in
the original document, and download the matching .docx summary. No further
notebook steps are needed for day-to-day use — this notebook is for
provisioning and redeploys only.